# Biomass Change — S3 in, visualization out

Pulls aboveground-biomass (AGB) rasters from a MAAP **S3 bucket through `maap-py`** and visualizes
the **change in biomass over a region** between two years.

1. **Locate** the two annual [ESA CCI Biomass v4](https://climate.esa.int/en/projects/biomass/) tiles
   (STAC Items) with [`pystac-client`](https://pystac-client.readthedocs.io/).
2. **Access S3** — turn each item's `s3://nasa-maap-data-store/...` asset into a presigned HTTPS URL
   with [`maap-py`](https://github.com/MAAP-Project/maap-py) (`maap.aws.s3_signed_url`).
3. **Read** the requested region from both years with [`rasterio`](https://rasterio.readthedocs.io/)
   (windowed read, no full-tile download).
4. **Visualize** the per-pixel difference (`AGB_end - AGB_start`) as a diverging map with
   [`matplotlib`](https://matplotlib.org/), written to `output/`.


In [ ]:
# Papermill parameters cell -- values here are overridden at execution time.
collection_id = "ESACCI_Biomass_L4_AGB_V4_100m"
tile_id = "S10W070"
year_start = 2019
year_end = 2020
asset_name = "estimates"
bbox = "-65 -12 -63 -10"  # MINX MINY MAXX MAXY, EPSG:4326
output_file = "biomass_change.png"
stac_url = "https://stac.maap-project.org/"

In [ ]:
import os
from urllib.parse import urlparse

import matplotlib

matplotlib.use("Agg")  # headless: render to file, no display needed
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import TwoSlopeNorm
from maap.maap import MAAP
from pystac_client import Client
from rasterio.transform import array_bounds
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

OUTPUT_DIR = "output"

# ESA CCI Biomass v4 STAC item ids are deterministic per tile + year.
ITEM_ID_TEMPLATE = "{tile}_ESACCI-BIOMASS-L4-AGB-MERGED-100m-{year}-fv4.0"

## Access the S3 asset through maap-py

ESA CCI assets live at `s3://nasa-maap-data-store/...`. Rather than juggling AWS credentials, ask
`maap-py` for a **presigned HTTPS URL** (`maap.aws.s3_signed_url`), which `rasterio`/GDAL can open
directly. `MAAP()` picks up the MAAP token and AWS credentials injected by the MAAP environment, so
no token or host needs to be supplied here.

In [ ]:
def signed_href(maap, s3_href):
    """Turn an ``s3://bucket/key`` href into a presigned HTTPS URL via maap-py."""
    if not s3_href.startswith("s3://"):
        raise ValueError(f"Expected an s3:// href, got: {s3_href}")
    parsed = urlparse(s3_href)
    bucket, key = parsed.netloc, parsed.path.lstrip("/")
    result = maap.aws.s3_signed_url(bucket, key)
    return result["url"]

## Read a region from each year

The bbox (EPSG:4326) is reprojected to the raster CRS and used to window-read just that region, so
only the bytes covering the area of interest are fetched. Both years share the same ESA CCI grid, so
the two windows align pixel-for-pixel.

In [ ]:
def read_agb_window(url, bbox):
    """Window-read band 1 of ``url`` over ``bbox`` (lon/lat); return (data, transform, nodata)."""
    with rasterio.open(url) as src:
        dst_bounds = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(*dst_bounds, transform=src.transform)
        window = window.round_offsets().round_lengths()
        if window.width <= 0 or window.height <= 0:
            raise ValueError(
                f"bbox {bbox} does not intersect the raster bounds {src.bounds}."
            )
        data = src.read(1, window=window).astype("float32")
        transform = src.window_transform(window)
        nodata = src.nodata
    return data, transform, nodata


def get_year_agb(maap, catalog, collection_id, tile_id, year, asset_name, bbox):
    """Find the {tile, year} STAC item, sign its asset, and window-read the bbox."""
    item_id = ITEM_ID_TEMPLATE.format(tile=tile_id, year=year)
    items = list(catalog.search(collections=[collection_id], ids=[item_id]).items())
    if not items:
        raise LookupError(
            f"No STAC item {item_id!r} in collection {collection_id!r}."
        )
    item = items[0]
    if asset_name not in item.assets:
        raise KeyError(
            f"Asset {asset_name!r} not found. Available: {sorted(item.assets)}"
        )
    url = signed_href(maap, item.assets[asset_name].href)
    data, transform, nodata = read_agb_window(url, bbox)
    print(f"{year}: read {data.shape} from {item_id}")
    return data, transform, nodata, item

## Compute the change

Mask nodata in either year, align the two arrays to a common shape, and difference them:
`ΔAGB = AGB_end - AGB_start` (Mg/ha). Positive = biomass gain, negative = loss.

In [ ]:
def compute_change(a_start, a_end, nd_start, nd_end):
    """Return ``a_end - a_start`` (Mg/ha) with nodata masked to NaN, arrays aligned."""
    # Crop to a common shape in case the two windows differ by a pixel.
    h = min(a_start.shape[0], a_end.shape[0])
    w = min(a_start.shape[1], a_end.shape[1])
    a_start = a_start[:h, :w].copy()
    a_end = a_end[:h, :w].copy()
    for arr, nd in ((a_start, nd_start), (a_end, nd_end)):
        if nd is not None:
            arr[arr == nd] = np.nan
    delta = a_end - a_start
    delta[np.isnan(a_start) | np.isnan(a_end)] = np.nan
    return delta

## Visualize the change

A **diverging** map centered at zero (`TwoSlopeNorm`) with the `BrBG` colormap: brown = biomass
**loss**, green = **gain**, near-neutral at no change. Brown/green is used instead of red/green so the
map stays readable for colorblind viewers. Colorbar limits are symmetric around zero (robust 98th
percentile) so gains and losses are comparable at a glance.

In [ ]:
def plot_change(delta, extent, tile_id, year_start, year_end, output_path):
    """Render the diverging ΔAGB map to ``output_path`` and return summary stats."""
    finite = delta[np.isfinite(delta)]
    lim = float(np.percentile(np.abs(finite), 98)) if finite.size else 1.0
    lim = lim or 1.0  # guard against an all-zero window
    norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(delta, cmap="BrBG", norm=norm, extent=extent, origin="upper")
    cbar = fig.colorbar(im, ax=ax, shrink=0.85)
    cbar.set_label(f"Apparent \u0394AGB, {year_start}\u2192{year_end} (Mg/ha)")

    mean_delta = float(np.nanmean(delta)) if finite.size else 0.0
    loss_pct = 100.0 * float(np.mean(finite < 0)) if finite.size else 0.0
    ax.set_title(
        f"ESA CCI aboveground biomass change\n"
        f"tile {tile_id}, {year_start}\u2192{year_end}"
    )
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.annotate(
        f"mean \u0394AGB = {mean_delta:+.1f} Mg/ha\narea with loss = {loss_pct:.0f}%",
        xy=(0.02, 0.02),
        xycoords="axes fraction",
        ha="left",
        va="bottom",
        fontsize=9,
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.85),
    )
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return mean_delta, loss_pct

## Run the workflow

In [ ]:
bbox_values = [float(v) for v in bbox.split()]
if len(bbox_values) != 4:
    raise ValueError("bbox must have four values: 'MINX MINY MAXX MAXY'")

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, output_file)

maap = MAAP()  # credentials injected by the MAAP environment
catalog = Client.open(stac_url)

a_start, transform, nd_start, _ = get_year_agb(
    maap, catalog, collection_id, tile_id, year_start, asset_name, bbox_values
)
a_end, _, nd_end, _ = get_year_agb(
    maap, catalog, collection_id, tile_id, year_end, asset_name, bbox_values
)

delta = compute_change(a_start, a_end, nd_start, nd_end)

# Geographic extent (lon/lat) of the differenced array for the map axes.
h, w = delta.shape
left, bottom, right, top = array_bounds(h, w, transform)
extent = (left, right, bottom, top)

mean_delta, loss_pct = plot_change(
    delta, extent, tile_id, year_start, year_end, output_path
)

print(f"\nWrote visualization to {output_path}")
print(f"mean apparent \u0394AGB = {mean_delta:+.1f} Mg/ha; area with loss = {loss_pct:.0f}%")